In [4]:
import pandas as pd
import numpy as np
df=pd.read_csv(r"C:\Users\Mohammed Abuzar\Downloads\PC11_LATRINES.csv", header=2)
df.head()

,state_code,state_name,National_rural_urban,total_hh_2011,total_hh_2001,water_closet_2011,water_closet_2001,pit_latrine_2011,pit_latrine_2001,other_latrine_2011,other_latrine_2001,no_latrine_2011,no_latrine_2001
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0.0,INDIA,National,246692667.0,191963935.0,36.4,18.0,9.4,11.5,1.1,6.9,53.1,63.6
2,1.0,Jammu & Kashmir,National,2015088.0,1551768.0,33.0,8.8,5.5,17.4,12.7,26.9,48.8,46.9
3,2.0,Himachal Pradesh,National,1476581.0,1240633.0,60.7,11.4,8.1,14.6,0.3,7.4,30.9,66.6
4,3.0,Punjab,National,5409699.0,4265156.0,59.3,20.4,19.2,24.3,0.8,12.1,20.7,43.2


In [6]:
print(df.columns)

Index(['state_code', 'state_name', 'National_rural_urban', 'total_hh_2011',
       'total_hh_2001', 'water_closet_2011', 'water_closet_2001',
       'pit_latrine_2011', 'pit_latrine_2001', 'other_latrine_2011',
       'other_latrine_2001', 'no_latrine_2011', 'no_latrine_2001'],
      dtype='object')


In [7]:
df.drop([
    'state_code',
    'National_rural_urban',
    'total_hh_2011',
    'total_hh_2001',
    'water_closet_2001',
    'pit_latrine_2001',
    'other_latrine_2001',
    'no_latrine_2001'
], axis=1, inplace=True)

In [8]:
print(df.columns)

Index(['state_name', 'water_closet_2011', 'pit_latrine_2011',
       'other_latrine_2011', 'no_latrine_2011'],
      dtype='object')


In [9]:
df.isnull().sum()

state_name            1
water_closet_2011     1
pit_latrine_2011      1
other_latrine_2011    1
no_latrine_2011       1
dtype: int64

In [10]:
numeric_cols = df.select_dtypes(include=np.number).columns
categorical_cols = df.select_dtypes(exclude=np.number).columns
print(numeric_cols)
print(categorical_cols)

Index(['water_closet_2011', 'pit_latrine_2011', 'other_latrine_2011',
       'no_latrine_2011'],
      dtype='object')
Index(['state_name'], dtype='object')


In [11]:
df.dtypes

state_name             object
water_closet_2011     float64
pit_latrine_2011      float64
other_latrine_2011    float64
no_latrine_2011       float64
dtype: object

In [12]:
for col in numeric_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].median())

In [13]:
for col in categorical_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].mode()[0])

In [14]:
df.isnull().sum()

state_name            0
water_closet_2011     0
pit_latrine_2011      0
other_latrine_2011    0
no_latrine_2011       0
dtype: int64

In [15]:
df.dropna(inplace=True)

In [16]:
from sklearn.impute import SimpleImputer
mean_imputer = SimpleImputer(strategy="mean")
df[numeric_cols] = mean_imputer.fit_transform(df[numeric_cols])

In [17]:
cat_imputer = SimpleImputer(strategy="most_frequent")
df[categorical_cols] = cat_imputer.fit_transform(df[categorical_cols])

In [18]:
df.isnull().sum()

state_name            0
water_closet_2011     0
pit_latrine_2011      0
other_latrine_2011    0
no_latrine_2011       0
dtype: int64

In [19]:
duplicate_count = df.duplicated().sum()
print(duplicate_count)

0


In [20]:
for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1- 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
outliers

,state_name,water_closet_2011,pit_latrine_2011,other_latrine_2011,no_latrine_2011


In [21]:
outliers.index

Index([], dtype='int64')

In [22]:
df.drop(outliers.index,axis=0,inplace=True)

In [25]:
df['Toilet_Access'] = (
    df['water_closet_2011'] +
    df['pit_latrine_2011'] +
    df['other_latrine_2011']
)

df['Open_Defecation'] = df['no_latrine_2011']

In [26]:
df['Year'] = 2019

In [27]:
df.rename(columns={'state_name': 'State'}, inplace=True)

df['State'] = df['State'].str.upper()

In [28]:
df = df[df['State'] != 'INDIA']

In [29]:
df = df[['State', 'Year', 'Toilet_Access', 'Open_Defecation']]

In [30]:
print(df.head())
print(df.shape)

              State  Year  Toilet_Access  Open_Defecation
0     A & N ISLANDS  2019          63.05            29.75
2   JAMMU & KASHMIR  2019          51.20            48.80
3  HIMACHAL PRADESH  2019          69.10            30.90
4            PUNJAB  2019          79.30            20.70
5        CHANDIGARH  2019          87.70            12.40
(106, 4)


In [31]:
df.to_csv(r"C:\AIML waterbourne project\New folder\cleaned_data.csv", index=False)